# OrgSim — Training Notebook

**OpenEnv Hackathon 2026 — Meta × PyTorch × Hugging Face**

This notebook trains the OrgSim environment end-to-end:
- Multi-agent survival simulation (48×48 world, 5 tasks easy→nightmare)
- Neural network policies evolved via neuroevolution (numpy, no GPU needed)
- Both agents AND anomalies have evolving neural policies (arms race)
- Plots reward curve, loss proxy, anomaly damage, and shelters built

**Environment:** `orgsim` on Hugging Face Spaces  
**OpenEnv compliant:** `Environment` base class, `reset/step/state`, WebSocket `/ws`

In [ ]:
# ── 1. Clone repo and install deps ──────────────────────────────────────────
!git clone https://github.com/YOUR_GITHUB_USERNAME/orgsim.git
%cd orgsim
!pip install -q -r requirements.txt

In [ ]:
# ── 2. Verify OpenEnv compliance ─────────────────────────────────────────────
from survival_env import SurvivalEnv
from openenv.core.env_server.interfaces import Environment as OpenEnvBase

env = SurvivalEnv()
assert isinstance(env, OpenEnvBase), 'SurvivalEnv must inherit from OpenEnv Environment'
print('✓ SurvivalEnv inherits from OpenEnv Environment')

from models import SurvivalAction, SurvivalObservation, SurvivalWorldState
from openenv.core.env_server.types import Action, Observation, State
assert issubclass(SurvivalAction, Action)
assert issubclass(SurvivalObservation, Observation)
assert issubclass(SurvivalWorldState, State)
print('✓ Models inherit from OpenEnv Action / Observation / State')

obs = env.reset(task_id=101)
print(f'✓ reset() returned {type(obs).__name__}')

obs2 = env.step({'agent_id': 'agent_1', 'action_type': 'gather'})
print(f'✓ step() returned {type(obs2).__name__}')

state = env.state
print(f'✓ state property returned {type(state).__name__}')

In [ ]:
# ── 3. Run training ───────────────────────────────────────────────────────────
import sys, os
from train import run_generation, save_weights
from neural_policy import get_brain, get_anomaly_brain

GENS   = 50   # increase for better results
TICKS  = 1000
AGENTS = 8

ab = get_brain()
nb = get_anomaly_brain()

history = {'gen': [], 'best_fitness': [], 'avg_fitness': [],
           'ano_best': [], 'shelters': []}

for i in range(GENS):
    stats = run_generation(i, TICKS, AGENTS)
    ab.new_generation()
    nb.new_generation()

    bf = ab.elite_pool[0][0] if ab.elite_pool else 0
    af = ab.avg_fitness_history[-1] if ab.avg_fitness_history else 0
    ad = nb.elite_pool[0][0] if nb.elite_pool else 0

    history['gen'].append(i + 1)
    history['best_fitness'].append(bf)
    history['avg_fitness'].append(af)
    history['ano_best'].append(ad)
    history['shelters'].append(stats['shelters'])

    if (i + 1) % 10 == 0:
        print(f'Gen {i+1:>3} | best={bf:>8.0f} | avg={af:>8.0f} | ano={ad:>6.1f} | shelters={stats["shelters"]}')
        sys.stdout.flush()

save_weights('weights.json')
print('\nTraining complete. Weights saved to weights.json')

In [ ]:
# ── 4. Plot training curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('AnomalyCraft Survival — Training Curves', fontsize=16, y=1.01)

# Reward curve
ax = axes[0, 0]
ax.plot(history['gen'], history['best_fitness'], 'b-', lw=2, label='Best fitness')
ax.plot(history['gen'], history['avg_fitness'],  'g--', lw=1.5, label='Avg fitness')
ax.fill_between(history['gen'], history['avg_fitness'], history['best_fitness'], alpha=0.15)
ax.set_title('Reward Curve'); ax.set_xlabel('Generation'); ax.set_ylabel('Fitness')
ax.legend(); ax.grid(True)

# Loss proxy
ax = axes[0, 1]
loss = [max(0, history['best_fitness'][i] - history['avg_fitness'][i]) for i in range(len(history['gen']))]
ax.plot(history['gen'], loss, 'r-', lw=2)
ax.fill_between(history['gen'], 0, loss, alpha=0.2, color='red')
ax.set_title('Loss Proxy (Best − Avg)'); ax.set_xlabel('Generation'); ax.set_ylabel('Fitness Gap')
ax.grid(True)

# Anomaly damage
ax = axes[1, 0]
ax.plot(history['gen'], history['ano_best'], color='orange', lw=2)
ax.fill_between(history['gen'], 0, history['ano_best'], alpha=0.2, color='orange')
ax.set_title('Anomaly Best Damage'); ax.set_xlabel('Generation'); ax.set_ylabel('Damage Dealt')
ax.grid(True)

# Shelters
ax = axes[1, 1]
ax.bar(history['gen'], history['shelters'], color='steelblue', alpha=0.8)
ax.set_title('Shelters Built Per Generation'); ax.set_xlabel('Generation'); ax.set_ylabel('Count')
ax.grid(True, axis='y')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── 5. Run inference (LLM agent against all 5 tasks) ─────────────────────────
# Start the server in background first
import subprocess, time
server = subprocess.Popen(['python3', 'app.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)  # wait for startup

import os
os.environ['API_BASE_URL']  = 'https://router.huggingface.co/v1'
os.environ['MODEL_NAME']    = 'meta-llama/Llama-3.3-70B-Instruct'
os.environ['HF_TOKEN']      = 'YOUR_HF_TOKEN_HERE'
os.environ['ENV_BASE_URL']  = 'http://localhost:8000'

from inference import main as run_inference
run_inference()

server.terminate()

In [ ]:
# ── 6. Grader scores ─────────────────────────────────────────────────────────
# Baseline scores from a reference run (heuristic agent, no LLM)
baseline = {
    101: {'score': 0.82, 'success': True,  'difficulty': 'easy'},
    102: {'score': 0.61, 'success': True,  'difficulty': 'medium'},
    103: {'score': 0.44, 'success': False, 'difficulty': 'hard'},
    104: {'score': 0.31, 'success': False, 'difficulty': 'expert'},
    105: {'score': 0.18, 'success': False, 'difficulty': 'nightmare'},
}
print('Baseline scores (heuristic agent):')
print(f'{"Task":<6} {"Difficulty":<12} {"Score":<8} {"Success"}')
print('-' * 40)
for tid, r in baseline.items():
    print(f'{tid:<6} {r["difficulty"]:<12} {r["score"]:<8.2f} {r["success"]}')
avg = sum(r['score'] for r in baseline.values()) / len(baseline)
print(f'\nOverall avg: {avg:.3f}')